# Generative AI — Saat Neural Network Mencipta, Bukan Hanya Mengenali

Sejauh ini di Module 02, semua model yang kamu bangun bersifat **diskriminatif**: diberi input, model menebak label atau angka.

- Notebook 01--02: mengklasifikasi gambar fashion ke 10 kategori
- Notebook 03: membedakan kucing vs anjing
- Notebook 04: memprediksi nilai berikutnya dari data berurutan

Di notebook ini kamu akan berkenalan dengan keluarga model yang berbeda: model **generatif** — model yang *menghasilkan data baru*, bukan sekadar menebak label. Inilah fondasi dari teknologi seperti ChatGPT dan Stable Diffusion.

## Tujuan Pembelajaran
1. Melatih **autoencoder**: jaringan yang belajar memampatkan dan merekonstruksi gambar
2. Menjelajah **latent space**: melihat digit "bermorfing" dari satu angka ke angka lain
3. Melatih **GAN (DCGAN) mini**: melihat digit MNIST "lahir" dari noise acak
4. Memahami peta dunia generatif modern: VAE, diffusion, dan GPT
5. Menjalankan **Stable Diffusion (SD-Turbo)** dan melihat proses denoising-nya selangkah demi selangkah

> **Prasyarat**: jalankan di Google Colab dengan GPU runtime (Runtime > Change runtime type > T4 GPU). Total waktu eksekusi di T4 kurang dari 10 menit (training AE+GAN ~5 menit; bagian diffusion menambah download model ~2.5 GB sekali).

In [ ]:
!nvidia-smi

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

print("TensorFlow:", tf.__version__)
print("GPU terdeteksi:", tf.config.list_physical_devices('GPU'))

# Seed di-fix supaya hasil kamu sama dengan yang dibahas di materi
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

## Bagian A — Autoencoder: Belajar Memampatkan, Lalu Mencipta Ulang

Autoencoder adalah neural network yang tugasnya terdengar aneh: **merekonstruksi inputnya sendiri**. Strukturnya seperti jam pasir:

- **Encoder**: memampatkan gambar 28x28 (784 angka) menjadi vektor kecil, misalnya 32 angka
- **Latent space**: "ruang ringkasan" berukuran 32 dimensi itu — esensi dari gambar
- **Decoder**: membangun ulang gambar 28x28 dari 32 angka tadi

Analogi: kamu memfoto sebuah lukisan, lalu mendeskripsikannya dalam satu kalimat (encoder), kemudian temanmu melukis ulang hanya dari kalimat itu (decoder). Kalau hasil lukisannya mirip, berarti kalimat ringkasannya bagus.

Kenapa ini penting untuk generative AI? Karena **decoder pada dasarnya adalah generator**: dia mengubah vektor angka menjadi gambar.

In [ ]:
# MNIST: 70.000 gambar digit tulisan tangan 28x28 grayscale
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalisasi ke 0.0-1.0 (pola yang sama seperti notebook 01)
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

print("Data train:", x_train.shape)
print("Data test :", x_test.shape)

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(14, 2))
for i, ax in enumerate(axes):
    ax.imshow(x_train[i], cmap="gray")
    ax.set_title(y_train[i])
    ax.axis("off")
plt.suptitle("10 sample digit MNIST")
plt.show()

### Membangun Autoencoder

Kita pakai `Dense` layer biasa (seperti notebook 01) supaya fokusnya ke *konsep*, bukan arsitektur:

- Encoder: 784 -> 128 -> **32** (latent)
- Decoder: 32 -> 128 -> 784 -> reshape ke 28x28

Loss-nya `binary_crossentropy` per pixel: seberapa beda gambar rekonstruksi dengan gambar asli. Perhatikan: **tidak ada label sama sekali** — targetnya adalah inputnya sendiri (`x_train, x_train`). Ini disebut *self-supervised*.

In [ ]:
LATENT_DIM = 32

encoder = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(LATENT_DIM, activation="relu"),
], name="encoder")

decoder = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation="relu", input_shape=(LATENT_DIM,)),
    tf.keras.layers.Dense(28 * 28, activation="sigmoid"),
    tf.keras.layers.Reshape((28, 28)),
], name="decoder")

autoencoder = tf.keras.Sequential([encoder, decoder], name="autoencoder")
autoencoder.compile(optimizer="adam", loss="binary_crossentropy")
autoencoder.summary()

In [ ]:
EPOCHS_AE = 10  # ~1 menit di T4

history = autoencoder.fit(
    x_train, x_train,          # input = target: rekonstruksi diri sendiri
    epochs=EPOCHS_AE,
    batch_size=256,
    validation_data=(x_test, x_test),
)

### Seberapa Bagus Rekonstruksinya?

Baris atas = gambar asli, baris bawah = hasil rekonstruksi setelah dipampatkan jadi 32 angka. Detailnya sedikit hilang (wajar — kompresi 784 jadi 32, sekitar 24x lebih kecil), tapi digitnya masih jelas terbaca.

In [ ]:
recon = autoencoder.predict(x_test[:10])

fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for i in range(10):
    axes[0, i].imshow(x_test[i], cmap="gray"); axes[0, i].axis("off")
    axes[1, i].imshow(recon[i], cmap="gray"); axes[1, i].axis("off")
plt.suptitle("Atas: asli | Bawah: rekonstruksi dari 32 angka latent")
plt.show()

### Menjelajah Latent Space: Morphing Antar Digit

Ini bagian serunya. Setiap gambar kini punya "alamat" berupa 32 angka di latent space. Kalau kita ambil alamat digit A dan alamat digit B, lalu berjalan pelan-pelan dari A ke B *di dalam latent space* sambil men-decode setiap langkahnya — kita akan melihat digit A **bermorfing** menjadi digit B.

Ini bukti bahwa latent space bukan sekadar kompresi: dia menyimpan *struktur* — titik-titik di antara dua digit valid juga menghasilkan gambar yang masuk akal.

In [ ]:
# Ambil satu digit '7' dan satu digit '2' dari test set
idx_a = int(np.where(y_test == 7)[0][0])
idx_b = int(np.where(y_test == 2)[0][0])

z_a = encoder(x_test[idx_a:idx_a + 1]).numpy()
z_b = encoder(x_test[idx_b:idx_b + 1]).numpy()

STEPS = 10
fig, axes = plt.subplots(1, STEPS, figsize=(15, 2))
for i, t in enumerate(np.linspace(0.0, 1.0, STEPS)):
    z = (1 - t) * z_a + t * z_b      # campuran linear dua "alamat"
    img = decoder(z).numpy()[0]
    axes[i].imshow(img, cmap="gray")
    axes[i].axis("off")
plt.suptitle("Interpolasi latent space: 7 bermorfing menjadi 2")
plt.show()

### Bisakah Kita Mencipta Digit Baru dari Nol?

Logikanya: kalau decoder bisa mengubah 32 angka menjadi gambar, kita tinggal kasih 32 angka **acak** dan dapat digit baru, kan? Coba kita buktikan.

In [ ]:
# Skala 3.0 sengaja dipilih jauh dari distribusi latent data asli
z_random = np.abs(np.random.normal(size=(10, LATENT_DIM))).astype("float32") * 3.0
imgs = decoder(z_random).numpy()

fig, axes = plt.subplots(1, 10, figsize=(14, 2))
for i, ax in enumerate(axes):
    ax.imshow(imgs[i], cmap="gray")
    ax.axis("off")
plt.suptitle("Decode dari titik acak di latent space: hasilnya tidak meyakinkan")
plt.show()

Hasilnya kabur dan banyak yang tidak berbentuk digit. Kenapa?

Autoencoder hanya dilatih untuk merekonstruksi gambar yang *ada* — dia tidak pernah diminta memastikan bahwa **setiap** titik di latent space menghasilkan gambar yang bagus. Titik acak yang kita pilih kemungkinan besar jatuh di "daerah kosong" yang tidak pernah dilewati data training.

Untuk benar-benar *mencipta*, kita butuh model yang sejak awal dilatih menghasilkan data baru dari noise. Masuklah: **GAN**.

## Bagian B — GAN: Dua Jaringan yang Saling Berlomba

**GAN (Generative Adversarial Network)** terdiri dari dua jaringan yang dilatih sebagai lawan tanding:

- **Generator** = pemalsu lukisan: mengubah noise acak menjadi gambar palsu, berusaha menipu
- **Discriminator** = kurator galeri: melihat gambar dan menebak — asli (dari dataset) atau palsu (dari generator)?

Keduanya dilatih bersamaan dalam satu langkah, masing-masing dengan loss sendiri. Setiap kali discriminator makin jago membedakan, generator dipaksa membuat pemalsuan yang lebih meyakinkan — dan sebaliknya. Hasil akhir perlombaan ini: generator yang bisa menghasilkan gambar yang nyaris tidak bisa dibedakan dari asli.

Versi yang kita pakai adalah **DCGAN** (Deep Convolutional GAN): generator memakai `Conv2DTranspose` (kebalikan arah convolution — memperbesar gambar), discriminator memakai `Conv2D` biasa seperti CNN di notebook 03.

In [ ]:
# GAN dengan output tanh bekerja di rentang -1..1, bukan 0..1
N_GAN = 20000  # subset supaya training ~10 menit di T4
x_gan = x_train[:N_GAN].reshape(-1, 28, 28, 1) * 2.0 - 1.0

BATCH_SIZE = 256
NOISE_DIM = 100

dataset = (tf.data.Dataset.from_tensor_slices(x_gan)
           .shuffle(N_GAN, seed=SEED)
           .batch(BATCH_SIZE))
print("Jumlah gambar untuk training GAN:", N_GAN)

In [ ]:
def build_generator():
    return tf.keras.Sequential([
        tf.keras.layers.Dense(7 * 7 * 128, use_bias=False, input_shape=(NOISE_DIM,)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.LeakyReLU(),
        tf.keras.layers.Reshape((7, 7, 128)),
        tf.keras.layers.Conv2DTranspose(64, 5, strides=2, padding="same", use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.LeakyReLU(),
        tf.keras.layers.Conv2DTranspose(1, 5, strides=2, padding="same",
                                        activation="tanh"),
    ], name="generator")

generator = build_generator()
generator.summary()

In [ ]:
def build_discriminator():
    return tf.keras.Sequential([
        tf.keras.layers.Conv2D(64, 5, strides=2, padding="same",
                               input_shape=(28, 28, 1)),
        tf.keras.layers.LeakyReLU(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Conv2D(128, 5, strides=2, padding="same"),
        tf.keras.layers.LeakyReLU(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(1),   # logit: > 0 cenderung asli, < 0 cenderung palsu
    ], name="discriminator")

discriminator = build_discriminator()
discriminator.summary()

### Training Loop Adversarial

Beda dengan `model.fit()` biasa, GAN butuh training loop manual karena dua model dilatih bersamaan dalam satu langkah, masing-masing dengan loss dan optimizer sendiri:

1. Generator membuat gambar palsu dari noise
2. Discriminator menilai gambar asli DAN palsu; loss-nya = seberapa sering dia salah tebak
3. Generator di-update berdasarkan seberapa sukses dia menipu discriminator

Perhatikan dua optimizer terpisah — masing-masing jaringan hanya meng-update weight miliknya sendiri.

In [ ]:
bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)
gen_opt = tf.keras.optimizers.Adam(1e-4)
disc_opt = tf.keras.optimizers.Adam(1e-4)


@tf.function
def train_step(real_images):
    noise = tf.random.normal([tf.shape(real_images)[0], NOISE_DIM])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        fake_images = generator(noise, training=True)

        real_logits = discriminator(real_images, training=True)
        fake_logits = discriminator(fake_images, training=True)

        # Discriminator: asli harus ditebak 1, palsu harus ditebak 0
        disc_loss = (bce(tf.ones_like(real_logits), real_logits) +
                     bce(tf.zeros_like(fake_logits), fake_logits))
        # Generator: sukses kalau yang palsu ditebak 1 (berhasil menipu)
        gen_loss = bce(tf.ones_like(fake_logits), fake_logits)

    gen_grads = gen_tape.gradient(gen_loss, generator.trainable_variables)
    disc_grads = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
    gen_opt.apply_gradients(zip(gen_grads, generator.trainable_variables))
    disc_opt.apply_gradients(zip(disc_grads, discriminator.trainable_variables))
    return gen_loss, disc_loss

In [ ]:
EPOCHS_GAN = 15  # ~3 menit di T4; makin lama makin halus hasilnya

# Noise tetap: digit yang sama kita pantau perkembangannya antar epoch
fixed_noise = tf.random.normal([16, NOISE_DIM], seed=SEED)


def show_generated(epoch):
    imgs = generator(fixed_noise, training=False).numpy()
    imgs = (imgs + 1.0) / 2.0  # kembalikan dari -1..1 ke 0..1
    fig, axes = plt.subplots(4, 4, figsize=(5, 5))
    for i, ax in enumerate(axes.flat):
        ax.imshow(imgs[i, :, :, 0], cmap="gray")
        ax.axis("off")
    plt.suptitle(f"Hasil generator --- epoch {epoch}")
    plt.show()


for epoch in range(1, EPOCHS_GAN + 1):
    for batch in dataset:
        gen_loss, disc_loss = train_step(batch)
    print(f"Epoch {epoch:2d}/{EPOCHS_GAN} | gen_loss={gen_loss:.3f} "
          f"| disc_loss={disc_loss:.3f}")
    if epoch in (1, 5, 10, EPOCHS_GAN):
        show_generated(epoch)

### Membaca Hasilnya

Perhatikan progresnya: di epoch 1 hasilnya masih noise berbintik, lalu perlahan muncul coretan, dan di epoch 15 sebagian besar sudah berbentuk digit yang bisa dikenali — **semuanya digit baru yang tidak ada di dataset**, lahir murni dari noise.

Dua catatan jujur tentang GAN:

- Hasil 15 epoch masih kasar. GAN "produksi" dilatih ratusan epoch dengan banyak trik kestabilan.
- GAN terkenal **rewel**: kadang generator menemukan satu-dua jenis digit yang selalu lolos, lalu berhenti belajar variasi lain (disebut *mode collapse*). Kalau hasil kamu didominasi digit yang itu-itu saja, kamu baru saja menyaksikannya sendiri.

Justru kerewelannya ini yang mendorong riset ke arsitektur generatif yang lebih stabil — yang akan kita petakan sekarang.

## Bagian C — Peta Dunia Generative AI Modern

Autoencoder dan GAN adalah dua leluhur penting, tapi dunia generatif 2020-an dikuasai dua keluarga lain:

| Keluarga | Ide inti | Contoh terkenal |
|---|---|---|
| Autoencoder | Kompres lalu rekonstruksi; decoder = generator | Kompresi, denoising |
| **VAE** | Autoencoder yang latent space-nya DIATUR supaya bisa di-sampling | Generator wajah awal |
| **GAN** | Generator vs discriminator saling berlomba | Deepfake, StyleGAN |
| **Diffusion** | Belajar MEMBERSIHKAN noise bertahap; generate = denoise dari noise murni | Stable Diffusion, DALL-E 2/3, Midjourney |
| **Autoregressive Transformer** | Prediksi token berikutnya, berulang-ulang | GPT, ChatGPT, Llama |

Dua benang merah yang berlaku untuk semuanya:

1. **Semua generator belajar distribusi data** — "seperti apa sih data yang wajar?" — lalu sampling dari situ.
2. **Mencipta = transformasi dari sesuatu yang sederhana** (noise, atau teks sebelumnya) **menjadi data yang kompleks**, dipelajari dari jutaan contoh.

Yang menarik: ChatGPT pada dasarnya melakukan hal yang sama dengan DCGAN kamu barusan — menghasilkan data baru yang meniru distribusi data training — hanya saja datanya teks, dan arsitekturnya transformer.
Satu keluarga di tabel ini bisa langsung kita buktikan sekarang: diffusion.

## Bagian D — Diffusion: Melihat Gambar Lahir dari Noise, Selangkah demi Selangkah

DCGAN kamu tadi menghasilkan digit dari noise dalam **sekali lompatan**: noise masuk, gambar keluar. Diffusion mengambil rute yang berbeda — dan justru rute ini yang dipakai Stable Diffusion, DALL-E 2/3, dan Midjourney.

Idenya terdiri dari dua proses:

1. **Forward diffusion** (mudah, tanpa belajar): ambil gambar asli, tambahkan noise sedikit demi sedikit sampai jadi noise murni
2. **Reverse diffusion** (ini yang dipelajari model): balikkan prosesnya — mulai dari noise murni, bersihkan selangkah demi selangkah sampai muncul gambar

Kita lihat keduanya. Forward bisa kamu buktikan sendiri dalam beberapa baris kode; untuk reverse kita pakai model pretrained, karena melatihnya dari nol butuh waktu jauh lebih lama dari sesi ini.

In [ ]:
# Forward diffusion: menenggelamkan digit ke dalam noise secara bertahap
img = x_train[0]

fig, axes = plt.subplots(1, 8, figsize=(14, 2))
for i, t in enumerate(np.linspace(0.0, 1.0, 8)):
    noise = np.random.normal(size=img.shape)
    noisy = np.sqrt(1.0 - t) * img + np.sqrt(t) * noise
    axes[i].imshow(noisy, cmap="gray")
    axes[i].set_title(f"t={t:.2f}", fontsize=8)
    axes[i].axis("off")
plt.suptitle("Forward diffusion: gambar asli (kiri) tenggelam menjadi noise murni (kanan)")
plt.show()

Mudah, kan? Yang sulit adalah arah sebaliknya: dari panel paling kanan kembali ke paling kiri. Model diffusion dilatih jutaan kali untuk menebak "noise mana yang tadi ditambahkan" — sehingga saat inference dia bisa **mencicil pembersihan noise** sampai muncul gambar baru.

Kita pakai **SD-Turbo** (versi cepat Stable Diffusion dari Stability AI) lewat library `diffusers`. Download model sekitar 2.5 GB — hanya sekali, setelah itu tersimpan di cache runtime.

In [ ]:
!pip install -q diffusers accelerate

In [ ]:
import torch
from diffusers import AutoPipelineForText2Image

pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sd-turbo", torch_dtype=torch.float16, variant="fp16"
).to("cuda")

prompt = "a watercolor painting of a cat playing guitar"  # prompt bahasa Inggris: model dilatih dengan caption Inggris

frames = []

def simpan_langkah(pipeline, step, timestep, callback_kwargs):
    # Decode latent di tengah proses supaya kita bisa melihat denoising berjalan
    latents = callback_kwargs["latents"]
    with torch.no_grad():
        decoded = pipeline.vae.decode(
            latents / pipeline.vae.config.scaling_factor
        ).sample
    img = (decoded / 2 + 0.5).clamp(0, 1)[0].permute(1, 2, 0).float().cpu().numpy()
    frames.append(img)
    return callback_kwargs

hasil = pipe(
    prompt,
    num_inference_steps=4,
    guidance_scale=0.0,   # SD-Turbo memang dirancang tanpa guidance
    callback_on_step_end=simpan_langkah,
    callback_on_step_end_tensor_inputs=["latents"],
).images[0]

In [ ]:
fig, axes = plt.subplots(1, len(frames) + 1, figsize=(16, 4))
for i, f in enumerate(frames):
    axes[i].imshow(f)
    axes[i].set_title(f"step {i + 1}", fontsize=9)
    axes[i].axis("off")
axes[-1].imshow(hasil)
axes[-1].set_title("hasil akhir", fontsize=9)
axes[-1].axis("off")
plt.suptitle("Reverse diffusion: noise dibersihkan selangkah demi selangkah menjadi gambar")
plt.show()

Perhatikan kemiripannya dengan grid DCGAN kamu: sama-sama noise yang berubah menjadi gambar. Bedanya:

- DCGAN: transformasi **sekali jalan** yang dipelajari lewat perlombaan dua network — progres yang kamu lihat tadi adalah progres *antar epoch training*
- Diffusion: transformasi **bertahap dalam satu proses inference** — progres yang kamu lihat di atas terjadi setiap kali generate
- Diffusion juga **dikondisikan teks** (prompt), karena dilatih berpasangan gambar + caption

Stabilitas training inilah (tidak ada perlombaan yang bisa kolaps) salah satu alasan besar kenapa dunia generative image bergeser dari GAN ke diffusion.

## Ringkasan

Hari ini kamu sudah:

1. Melatih **autoencoder** dan membuktikan decoder bisa "melukis" dari 32 angka
2. Melihat **latent space** menyimpan struktur (interpolasi 7 -> 2 yang mulus)
3. Membuktikan kenapa AE biasa belum cukup untuk mencipta (sampling acak gagal)
4. Melatih **DCGAN** dan menyaksikan digit lahir dari noise lewat perlombaan generator vs discriminator
5. Memetakan keluarga generatif modern: VAE, diffusion, dan autoregressive transformer
6. Menjalankan **reverse diffusion** sungguhan dengan SD-Turbo dan melihat noise menjadi gambar selangkah demi selangkah

**Menuju Module 04**: di sana kamu akan memakai model generatif untuk *teks* — LLM seperti GPT. Bekal dari notebook ini: LLM juga generator yang belajar distribusi data; bedanya dia men-generate token demi token, bukan pixel.